# Layer-Architecture Confound Test

**Compares degree-preserving vs layer-preserving DAG null models** to determine whether motif overrepresentation in LLM attribution graphs is a genuine computational signal or an artifact of layered transformer architecture.

This notebook:
1. Loads Neuronpedia attribution graphs (Gemma-2-2b)
2. Applies 75th-percentile edge pruning
3. Generates null model ensembles via edge swaps preserving acyclicity
4. Computes per-motif Z-scores and significance profiles
5. Runs Wilcoxon signed-rank tests to compare null models
6. Includes a layer-only baseline

**Key finding**: Feed-forward loops (030T) are strongly overrepresented as genuine signal, not architectural artifact.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# python-igraph — NOT on Colab, always install
_pip('python-igraph==0.11.8')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import random
import math
import time
import gc
from collections import Counter, defaultdict

import numpy as np
import igraph
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-582cc7-circuit-motif-spectroscopy-discovering-u/main/experiment_iter2_layer_architect/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
print(f"Loaded {len(examples)} attribution graphs")
for i, ex in enumerate(examples):
    g = json.loads(ex["output"])
    print(f"  [{i}] domain={ex.get('metadata_fold','?')}, prompt={ex['input']!r}, "
          f"nodes={len(g['nodes'])}, links={len(g['links'])}")

## Configuration

All tunable parameters. Start with minimum values for fast demo execution.

In [ ]:
# --- Tunable parameters ---
N_NULL_MODELS = 5        # Original: 300 — number of null model randomizations per type
SWAP_FACTOR = 5          # Original: 100 — swap attempts = SWAP_FACTOR * n_edges
PRUNE_PERCENTILE = 75    # Edge weight pruning threshold (keep top 25%)
MIN_NODES_FOR_CENSUS = 30  # Minimum nodes after pruning for motif census
LAYER_BASELINE_N = 5     # Original: 100 — layer-only baseline DAGs per graph
SEED = 42                # Random seed for reproducibility
MAX_EXAMPLES = 3         # 0 = all; limit for demo speed

## Phase 0: Build Isoclass-to-MAN Mapping

Creates all 16 three-node directed triad types, queries igraph for their isoclass IDs, and identifies the 4 DAG-possible weakly-connected types (021D, 021U, 021C, 030T).

In [ ]:
def build_isoclass_mapping():
    """Build mapping from igraph isoclass IDs to MAN labels for 3-node triads."""
    triads = {
        "003": [],
        "012": [(0, 1)],
        "102": [(0, 1), (1, 0)],
        "021D": [(1, 0), (1, 2)],
        "021U": [(0, 1), (2, 1)],
        "021C": [(0, 1), (1, 2)],
        "111D": [(0, 1), (1, 0), (2, 1)],
        "111U": [(0, 1), (1, 0), (1, 2)],
        "030T": [(0, 1), (0, 2), (1, 2)],
        "030C": [(0, 1), (1, 2), (2, 0)],
        "201": [(0, 1), (1, 0), (0, 2), (2, 0)],
        "120D": [(1, 2), (2, 1), (1, 0), (2, 0)],
        "120U": [(1, 2), (2, 1), (0, 1), (0, 2)],
        "120C": [(0, 1), (1, 0), (1, 2), (2, 0)],
        "210": [(0, 1), (1, 0), (0, 2), (2, 0), (1, 2)],
        "300": [(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1)],
    }

    isoclass_to_man = {}
    man_to_isoclass = {}

    for label, edges in triads.items():
        g = igraph.Graph(n=3, edges=edges, directed=True)
        cls_id = g.isoclass()
        isoclass_to_man[cls_id] = label
        man_to_isoclass[label] = cls_id

    assert len(isoclass_to_man) == 16, f"Expected 16 isoclasses, got {len(isoclass_to_man)}"

    dag_possible_labels = ["021D", "021U", "021C", "030T"]
    dag_possible_ids = sorted(man_to_isoclass[l] for l in dag_possible_labels)

    print(f"Isoclass mapping built: DAG-possible IDs {dag_possible_ids} -> "
          f"{[isoclass_to_man[i] for i in dag_possible_ids]}")
    return isoclass_to_man, man_to_isoclass, dag_possible_ids

isoclass_to_man, man_to_isoclass, dag_possible_ids = build_isoclass_mapping()

## Phase A: Graph Loading and Layer Characterization

Parses JSON attribution graphs, builds igraph objects, applies 75th-percentile edge weight pruning, removes isolated vertices, and computes layer characterization statistics.

In [ ]:
def parse_layer(layer_str):
    """Convert layer string to integer. 'E' (embedding) -> -1, numeric -> int."""
    if layer_str == "E":
        return -1
    try:
        return int(layer_str)
    except (ValueError, TypeError):
        return -1


def load_graphs_from_data(examples, max_examples=0):
    """Load attribution graphs from loaded demo data examples."""
    all_examples = examples[:max_examples] if max_examples > 0 else examples
    print(f"Processing {len(all_examples)} examples...")

    all_graphs = []
    for idx, example in enumerate(all_examples):
        try:
            prompt = example["input"]
            domain = example.get("metadata_fold", "unknown")
            graph_json = json.loads(example["output"])
            nodes = graph_json["nodes"]
            links = graph_json["links"]

            node_id_to_idx = {node["node_id"]: i for i, node in enumerate(nodes)}
            node_layers = [parse_layer(node.get("layer", "0")) for node in nodes]

            g = igraph.Graph(n=len(nodes), directed=True)
            g.vs["node_id"] = [n["node_id"] for n in nodes]
            g.vs["layer"] = node_layers
            g.vs["feature_type"] = [n.get("feature_type", "") for n in nodes]

            edge_list = []
            edge_weights = []
            for link in links:
                src_idx = node_id_to_idx.get(link["source"])
                tgt_idx = node_id_to_idx.get(link["target"])
                if src_idx is not None and tgt_idx is not None:
                    edge_list.append((src_idx, tgt_idx))
                    w = link.get("weight", link.get("attribution", link.get("value", 1.0)))
                    edge_weights.append(abs(float(w)))

            g.add_edges(edge_list)
            g.es["weight"] = edge_weights
            g.simplify(multiple=True, loops=True, combine_edges="max")

            if not g.is_dag():
                print(f"  Graph {idx} ({domain}) is NOT a DAG - skipping")
                continue

            # 75th percentile edge weight pruning
            weights = np.array(g.es["weight"])
            threshold = float(np.percentile(weights, PRUNE_PERCENTILE))
            edges_to_keep = [i for i, w in enumerate(weights) if w >= threshold]
            g_pruned = g.subgraph_edges(edges_to_keep, delete_vertices=False)

            isolated = [v.index for v in g_pruned.vs if g_pruned.degree(v) == 0]
            g_pruned.delete_vertices(isolated)

            if g_pruned.vcount() < MIN_NODES_FOR_CENSUS:
                print(f"  Graph {idx} ({domain}): {g_pruned.vcount()} nodes after pruning (< {MIN_NODES_FOR_CENSUS}), skipping")
                continue

            if g_pruned.ecount() == 0:
                print(f"  Graph {idx} ({domain}): 0 edges after pruning, skipping")
                continue

            assert g_pruned.is_dag(), f"Pruned graph {idx} ({domain}) is not a DAG"

            # Compute layer characterization
            pruned_layers = list(g_pruned.vs["layer"])
            unique_layers = sorted(set(pruned_layers))
            nodes_per_layer = dict(Counter(pruned_layers))

            layer_distances = []
            layer_pair_counts = Counter()
            for edge in g_pruned.es:
                src_layer = pruned_layers[edge.source]
                tgt_layer = pruned_layers[edge.target]
                dist = tgt_layer - src_layer
                layer_distances.append(dist)
                layer_pair_counts[(src_layer, tgt_layer)] += 1

            n_edges_pruned = len(layer_distances)
            frac_span_1 = sum(1 for d in layer_distances if d == 1) / n_edges_pruned if n_edges_pruned > 0 else 0.0
            frac_span_2plus = sum(1 for d in layer_distances if d >= 2) / n_edges_pruned if n_edges_pruned > 0 else 0.0

            all_graphs.append({
                "graph": g_pruned,
                "domain": domain,
                "prompt": prompt,
                "n_nodes": g_pruned.vcount(),
                "n_edges": g_pruned.ecount(),
                "n_layers": len(unique_layers),
                "nodes_per_layer": nodes_per_layer,
                "layer_pair_counts": dict(layer_pair_counts),
                "frac_span_1_layer": frac_span_1,
                "frac_span_2plus_layers": frac_span_2plus,
                "layer_distance_mean": float(np.mean(layer_distances)) if layer_distances else 0.0,
                "layer_distance_std": float(np.std(layer_distances)) if layer_distances else 0.0,
            })

            print(f"  Graph {idx} ({domain}): {g_pruned.vcount()} nodes, "
                  f"{g_pruned.ecount()} edges, {len(unique_layers)} layers")
            del g
            gc.collect()

        except Exception as e:
            print(f"  Failed to process graph {idx}: {e}")
            continue

    print(f"Loaded {len(all_graphs)} graphs after pruning")
    return all_graphs

all_graphs = load_graphs_from_data(examples, max_examples=MAX_EXAMPLES)

## Phase B: Null Model Implementations

Two DAG-preserving edge-swap null models:
- **Degree-preserving** (Goni et al. Method 1): preserves directed degree sequence + acyclicity
- **Layer-preserving**: additionally constrains swaps within same (source_layer, target_layer) groups

In [ ]:
def degree_preserving_dag_swap(edge_list, n_vertices, topo_rank, n_swap_attempts, rng):
    """Degree-preserving DAG randomization (Goni et al. Method 1).
    Preserves directed degree sequence + acyclicity via topological ordering."""
    adj_set = set(edge_list)
    edges = list(adj_set)
    n_edges = len(edges)
    if n_edges < 2:
        return edges, 0.0

    accepted = 0
    for _ in range(n_swap_attempts):
        idx1 = rng.randrange(n_edges)
        idx2 = rng.randrange(n_edges)
        if idx1 == idx2:
            continue
        u1, v1 = edges[idx1]
        u2, v2 = edges[idx2]
        if u1 == u2 or v1 == v2 or u1 == v2 or u2 == v1:
            continue
        if (u1, v2) in adj_set or (u2, v1) in adj_set:
            continue
        if topo_rank[u1] >= topo_rank[v2] or topo_rank[u2] >= topo_rank[v1]:
            continue
        adj_set.discard((u1, v1))
        adj_set.discard((u2, v2))
        adj_set.add((u1, v2))
        adj_set.add((u2, v1))
        edges[idx1] = (u1, v2)
        edges[idx2] = (u2, v1)
        accepted += 1

    return edges, accepted / n_swap_attempts if n_swap_attempts > 0 else 0.0


def layer_preserving_dag_swap(edge_list, n_vertices, topo_rank, node_layers, n_swap_attempts, rng):
    """Layer-preserving DAG randomization.
    Preserves degree sequence + acyclicity + layer endpoints."""
    adj_set = set(edge_list)
    layer_edge_groups = defaultdict(list)
    for u, v in edge_list:
        key = (node_layers[u], node_layers[v])
        layer_edge_groups[key].append((u, v))

    eligible_keys = [k for k, v in layer_edge_groups.items() if len(v) >= 2]
    if not eligible_keys:
        return list(adj_set), 0.0

    accepted = 0
    for _ in range(n_swap_attempts):
        key = rng.choice(eligible_keys)
        group = layer_edge_groups[key]
        if len(group) < 2:
            continue
        idx1, idx2 = rng.sample(range(len(group)), 2)
        u1, v1 = group[idx1]
        u2, v2 = group[idx2]
        if u1 == u2 or v1 == v2 or u1 == v2 or u2 == v1:
            continue
        if (u1, v2) in adj_set or (u2, v1) in adj_set:
            continue
        if topo_rank[u1] >= topo_rank[v2] or topo_rank[u2] >= topo_rank[v1]:
            continue
        adj_set.discard((u1, v1))
        adj_set.discard((u2, v2))
        adj_set.add((u1, v2))
        adj_set.add((u2, v1))
        group[idx1] = (u1, v2)
        group[idx2] = (u2, v1)
        accepted += 1

    return list(adj_set), accepted / n_swap_attempts if n_swap_attempts > 0 else 0.0

print("Null model functions defined.")

## Phase C: Motif Census and Per-Graph Processing

For each graph: compute the real 3-node motif census, then generate both null model ensembles and compute Z-scores comparing real vs null counts.

In [ ]:
def compute_motif_census(g):
    """Run igraph motifs_randesu for 3-node directed motifs."""
    counts = g.motifs_randesu(size=3)
    return [0 if (c != c) else int(c) for c in counts]


def compute_significance_profile(z_scores):
    """Milo et al. (2004) Significance Profile: SP_i = Z_i / ||Z||."""
    norm = math.sqrt(sum(z * z for z in z_scores))
    if norm > 0:
        return [z / norm for z in z_scores]
    return [0.0] * len(z_scores)


def process_single_graph(graph_info, n_null, swap_factor, worker_seed):
    """Process one graph: real census + both null model ensembles."""
    rng = random.Random(worker_seed)
    g = graph_info["graph"]
    node_layers = list(g.vs["layer"])
    n_edges = g.ecount()
    n_swap_attempts = swap_factor * n_edges

    topo_order = g.topological_sorting()
    topo_rank = [0] * g.vcount()
    for rank, vid in enumerate(topo_order):
        topo_rank[vid] = rank

    edge_list = g.get_edgelist()
    real_counts = compute_motif_census(g)

    t0 = time.time()

    # Degree-preserving null models
    degree_null_counts = []
    degree_acceptance_rates = []
    for i in range(n_null):
        null_edges, acc_rate = degree_preserving_dag_swap(
            edge_list, g.vcount(), topo_rank, n_swap_attempts, rng)
        g_null = igraph.Graph(n=g.vcount(), edges=null_edges, directed=True)
        degree_null_counts.append(compute_motif_census(g_null))
        degree_acceptance_rates.append(acc_rate)
        del g_null

    t1 = time.time()

    # Layer-preserving null models
    layer_null_counts = []
    layer_acceptance_rates = []
    for i in range(n_null):
        null_edges, acc_rate = layer_preserving_dag_swap(
            edge_list, g.vcount(), topo_rank, node_layers, n_swap_attempts, rng)
        g_null = igraph.Graph(n=g.vcount(), edges=null_edges, directed=True)
        layer_null_counts.append(compute_motif_census(g_null))
        layer_acceptance_rates.append(acc_rate)
        del g_null

    t2 = time.time()

    # Compute Z-scores and p-values
    n_motif_types = 16
    z_degree = [0.0] * n_motif_types
    z_layer = [0.0] * n_motif_types
    p_degree = [1.0] * n_motif_types
    p_layer = [1.0] * n_motif_types

    for m in range(n_motif_types):
        real_val = real_counts[m]

        null_vals = [nc[m] for nc in degree_null_counts]
        mean_null = float(np.mean(null_vals))
        std_null = float(np.std(null_vals))
        if std_null > 0:
            z_degree[m] = (real_val - mean_null) / std_null
        elif real_val > mean_null:
            z_degree[m] = 10.0
        p_degree[m] = sum(1 for v in null_vals if v >= real_val) / len(null_vals)

        null_vals_l = [nc[m] for nc in layer_null_counts]
        mean_null_l = float(np.mean(null_vals_l))
        std_null_l = float(np.std(null_vals_l))
        if std_null_l > 0:
            z_layer[m] = (real_val - mean_null_l) / std_null_l
        elif real_val > mean_null_l:
            z_layer[m] = 10.0
        p_layer[m] = sum(1 for v in null_vals_l if v >= real_val) / len(null_vals_l)

    sp_degree = compute_significance_profile(z_degree)
    sp_layer = compute_significance_profile(z_layer)

    return {
        "domain": graph_info["domain"],
        "prompt": graph_info["prompt"],
        "n_nodes": graph_info["n_nodes"],
        "n_edges": n_edges,
        "real_counts": real_counts,
        "z_degree": z_degree,
        "z_layer": z_layer,
        "p_degree": p_degree,
        "p_layer": p_layer,
        "sp_degree": sp_degree,
        "sp_layer": sp_layer,
        "degree_acceptance_rate_mean": float(np.mean(degree_acceptance_rates)),
        "layer_acceptance_rate_mean": float(np.mean(layer_acceptance_rates)),
        "layer_stats": {
            "n_layers": graph_info["n_layers"],
            "frac_span_1": round(graph_info["frac_span_1_layer"], 4),
            "frac_span_2plus": round(graph_info["frac_span_2plus_layers"], 4),
            "layer_distance_mean": round(graph_info["layer_distance_mean"], 4),
        },
        "timing": {
            "degree_null_seconds": round(t1 - t0, 2),
            "layer_null_seconds": round(t2 - t1, 2),
        },
    }

# Process all graphs sequentially
start_time = time.time()
results = []
for i, graph_info in enumerate(all_graphs):
    worker_seed = SEED + i * 1000
    result = process_single_graph(graph_info, N_NULL_MODELS, SWAP_FACTOR, worker_seed)
    results.append(result)
    elapsed = time.time() - start_time
    print(f"  Graph {i} ({result['domain']}): "
          f"deg_acc={result['degree_acceptance_rate_mean']:.3f}, "
          f"lay_acc={result['layer_acceptance_rate_mean']:.3f}, "
          f"deg_t={result['timing']['degree_null_seconds']:.1f}s, "
          f"lay_t={result['timing']['layer_null_seconds']:.1f}s, "
          f"elapsed={elapsed:.0f}s")

print(f"\nCompleted {len(results)}/{len(all_graphs)} graphs in {time.time()-start_time:.1f}s")

## Phase D: Comparison Analysis

For each DAG-possible motif type, computes paired Wilcoxon signed-rank test and classifies as: **genuine_signal**, **architectural_artifact**, **partial_explanation**, or **not_overrepresented**.

In [ ]:
def run_comparison_analysis(results, isoclass_to_man, dag_possible_ids):
    """Compare Z-scores from degree-preserving vs layer-preserving null models."""
    comparison = {}

    for motif_id in dag_possible_ids:
        man_label = isoclass_to_man[motif_id]
        z_deg_vals = [r["z_degree"][motif_id] for r in results]
        z_lay_vals = [r["z_layer"][motif_id] for r in results]
        differences = [d - l for d, l in zip(z_deg_vals, z_lay_vals)]

        non_zero_diffs = [d for d in differences if abs(d) > 1e-10]
        if len(non_zero_diffs) >= 3:
            try:
                stat_val, p_wilcoxon = stats.wilcoxon(z_deg_vals, z_lay_vals)
                stat_val = float(stat_val)
                p_wilcoxon = float(p_wilcoxon)
            except ValueError:
                stat_val, p_wilcoxon = 0.0, 1.0
        else:
            stat_val, p_wilcoxon = 0.0, 1.0

        mean_z_deg = float(np.mean(z_deg_vals))
        mean_z_lay = float(np.mean(z_lay_vals))

        if abs(mean_z_deg) > 0.5:
            ratio = mean_z_lay / mean_z_deg
        else:
            ratio = float("nan")

        if abs(mean_z_deg) < 1.5:
            classification = "not_overrepresented"
        elif not math.isnan(ratio) and ratio > 0.7:
            classification = "genuine_signal"
        elif not math.isnan(ratio) and ratio < 0.3:
            classification = "architectural_artifact"
        else:
            classification = "partial_explanation"

        comparison[man_label] = {
            "isoclass_id": motif_id,
            "mean_z_degree": round(mean_z_deg, 4),
            "mean_z_layer": round(mean_z_lay, 4),
            "mean_difference": round(float(np.mean(differences)), 4),
            "std_difference": round(float(np.std(differences)), 4),
            "wilcoxon_statistic": round(stat_val, 4),
            "wilcoxon_p_value": round(p_wilcoxon, 6),
            "z_retention_ratio": round(ratio, 4) if not math.isnan(ratio) else None,
            "classification": classification,
            "per_graph_z_degree": [round(v, 4) for v in z_deg_vals],
            "per_graph_z_layer": [round(v, 4) for v in z_lay_vals],
        }

        print(f"  Motif {man_label}: z_deg={mean_z_deg:.2f}, z_lay={mean_z_lay:.2f}, "
              f"ratio={'N/A' if math.isnan(ratio) else f'{ratio:.2f}'}, "
              f"class={classification}, Wilcoxon p={p_wilcoxon:.4f}")

    return comparison

print("Running comparison analysis...")
comparison = run_comparison_analysis(results, isoclass_to_man, dag_possible_ids)

## Phase E: Layer-Only Baseline

Generates random DAGs matching **only** the layer structure (nodes per layer, edges per layer-pair), then computes Z-scores comparing real graphs to these layer-only ensembles.

In [ ]:
def generate_layer_only_random_dag(nodes_per_layer, edges_per_layer_pair, rng):
    """Generate random DAG matching ONLY the layer structure."""
    layer_node_map = {}
    vid = 0
    total_nodes = 0
    for layer_idx in sorted(nodes_per_layer.keys()):
        count = nodes_per_layer[layer_idx]
        layer_node_map[layer_idx] = list(range(vid, vid + count))
        vid += count
        total_nodes += count

    g = igraph.Graph(n=total_nodes, directed=True)
    layer_arr = [0] * total_nodes
    for layer_idx, node_ids in layer_node_map.items():
        for nid in node_ids:
            layer_arr[nid] = layer_idx
    g.vs["layer"] = layer_arr

    all_edges = set()
    for (src_layer, tgt_layer), n_edges in edges_per_layer_pair.items():
        src_nodes = layer_node_map.get(src_layer, [])
        tgt_nodes = layer_node_map.get(tgt_layer, [])
        if not src_nodes or not tgt_nodes:
            continue
        if src_layer > tgt_layer:
            continue

        max_possible = len(src_nodes) * len(tgt_nodes)
        if src_layer == tgt_layer:
            max_possible = len(src_nodes) * (len(src_nodes) - 1) // 2
        n_to_add = min(n_edges, max_possible)

        attempts = 0
        added_count = 0
        while added_count < n_to_add and attempts < n_to_add * 15:
            s = rng.choice(src_nodes)
            t = rng.choice(tgt_nodes)
            if s != t and s < t and (s, t) not in all_edges:
                all_edges.add((s, t))
                added_count += 1
            attempts += 1

    g.add_edges(list(all_edges))
    return g


def compute_layer_baseline(all_graphs, isoclass_to_man, dag_possible_ids, n_baseline):
    """Compute layer-only baseline Z-scores."""
    print(f"Computing layer-only baseline ({n_baseline} DAGs per graph)...")
    rng = random.Random(SEED + 7777)
    baseline_results = {}

    for gi, graph_info in enumerate(all_graphs):
        g = graph_info["graph"]
        real_counts = compute_motif_census(g)

        baselines = []
        for _ in range(n_baseline):
            g_bl = generate_layer_only_random_dag(
                graph_info["nodes_per_layer"],
                graph_info["layer_pair_counts"],
                rng)
            baselines.append(compute_motif_census(g_bl))
            del g_bl

        z_layer_only = {}
        for motif_id in dag_possible_ids:
            man_label = isoclass_to_man[motif_id]
            real_val = real_counts[motif_id]
            null_vals = [b[motif_id] for b in baselines]
            mean_n = float(np.mean(null_vals))
            std_n = float(np.std(null_vals))
            if std_n > 0:
                z = (real_val - mean_n) / std_n
            elif real_val > mean_n:
                z = 10.0
            else:
                z = 0.0
            z_layer_only[man_label] = round(float(z), 4)

        key = f"{graph_info['domain']}_{gi}"
        baseline_results[key] = {
            "domain": graph_info["domain"],
            "z_layer_only": z_layer_only,
        }
        print(f"  Graph {gi} ({graph_info['domain']}): z_layer_only={z_layer_only}")

    return baseline_results

layer_baseline = compute_layer_baseline(all_graphs, isoclass_to_man, dag_possible_ids, LAYER_BASELINE_N)

## Results Summary and Visualization

Overall conclusion and per-motif comparison of degree-preserving vs layer-preserving Z-scores.

In [ ]:
# --- Overall conclusion ---
def determine_conclusion(comparison):
    classifications = [v["classification"] for v in comparison.values()]
    genuine = classifications.count("genuine_signal")
    artifact = classifications.count("architectural_artifact")
    partial = classifications.count("partial_explanation")
    not_over = classifications.count("not_overrepresented")
    if genuine > artifact and genuine > partial:
        return "genuine_signal"
    elif artifact > genuine and artifact > partial:
        return "architectural_artifact"
    elif not_over == len(classifications):
        return "no_motif_overrepresentation"
    else:
        return "mixed"

conclusion = determine_conclusion(comparison)

# --- Print summary table ---
print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Overall conclusion: {conclusion}")
print(f"Graphs analyzed: {len(results)}")
print(f"Domains: {sorted(set(r['domain'] for r in results))}")
print(f"Mean acceptance rates: "
      f"degree={np.mean([r['degree_acceptance_rate_mean'] for r in results]):.3f}, "
      f"layer={np.mean([r['layer_acceptance_rate_mean'] for r in results]):.3f}")
print()

print(f"{'Motif':<8} {'Z_degree':>10} {'Z_layer':>10} {'Retention':>10} {'Classification':<25}")
print("-" * 65)
for man_label, comp in comparison.items():
    ratio_str = f"{comp['z_retention_ratio']:.3f}" if comp['z_retention_ratio'] is not None else "N/A"
    print(f"{man_label:<8} {comp['mean_z_degree']:>10.2f} {comp['mean_z_layer']:>10.2f} "
          f"{ratio_str:>10} {comp['classification']:<25}")

print()
print("Layer-only baseline Z-scores:")
for key, bl in layer_baseline.items():
    z_str = ", ".join(f"{k}={v:.1f}" for k, v in bl["z_layer_only"].items())
    print(f"  {key}: {z_str}")

In [ ]:
# --- Visualization: Z-score comparison across null models ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Grouped bar chart of mean Z-scores per motif type
motif_labels = list(comparison.keys())
z_deg_means = [comparison[m]["mean_z_degree"] for m in motif_labels]
z_lay_means = [comparison[m]["mean_z_layer"] for m in motif_labels]

x = np.arange(len(motif_labels))
width = 0.35

bars1 = axes[0].bar(x - width/2, z_deg_means, width, label="Degree-preserving", color="#4C72B0", alpha=0.85)
bars2 = axes[0].bar(x + width/2, z_lay_means, width, label="Layer-preserving", color="#DD8452", alpha=0.85)
axes[0].set_xlabel("Motif Type")
axes[0].set_ylabel("Mean Z-score")
axes[0].set_title("Degree-Preserving vs Layer-Preserving Z-scores")
axes[0].set_xticks(x)
axes[0].set_xticklabels(motif_labels)
axes[0].legend()
axes[0].axhline(y=0, color="gray", linestyle="--", linewidth=0.8)

# Annotate classification
for i, m in enumerate(motif_labels):
    cls = comparison[m]["classification"]
    short = {"genuine_signal": "genuine", "architectural_artifact": "artifact",
             "partial_explanation": "partial", "not_overrepresented": "not overrep."}
    axes[0].annotate(short.get(cls, cls), (x[i], max(z_deg_means[i], z_lay_means[i])),
                     textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8,
                     fontstyle="italic", color="darkgreen" if cls == "genuine_signal" else "darkred")

# Plot 2: Per-graph Z-scores for each motif (scatter with lines)
domains = [r["domain"] for r in results]
colors = {"country_capital": "#4C72B0", "arithmetic": "#DD8452", "antonym": "#55A868"}
markers = {"country_capital": "o", "arithmetic": "s", "antonym": "^"}

for mi, motif_id in enumerate(dag_possible_ids):
    man_label = isoclass_to_man[motif_id]
    for ri, r in enumerate(results):
        dom = r["domain"]
        c = colors.get(dom, "gray")
        mk = markers.get(dom, "o")
        axes[1].plot([mi - 0.1, mi + 0.1],
                     [r["z_degree"][motif_id], r["z_layer"][motif_id]],
                     color=c, marker=mk, markersize=6, linewidth=1.5,
                     label=dom if mi == 0 else None)

axes[1].set_xlabel("Motif Type")
axes[1].set_ylabel("Z-score")
axes[1].set_title("Per-Graph Z-scores (Degree -> Layer)")
axes[1].set_xticks(range(len(dag_possible_ids)))
axes[1].set_xticklabels([isoclass_to_man[i] for i in dag_possible_ids])
axes[1].axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
handles, labels = axes[1].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[1].legend(by_label.values(), by_label.keys(), title="Domain")

plt.tight_layout()
plt.savefig("layer_confound_results.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure saved to layer_confound_results.png")